# Exploratory Data Analysis

This notebook explores the PhonePe data step by step before opportunity scoring.

The analysis answers five practical questions:

1. Is the latest data complete enough to compare markets?
2. How concentrated is payment demand?
3. Which states and districts show the strongest demand and growth?
4. Where is merchant penetration relatively low compared with demand?
5. Which districts repeatedly appear as stronger merchant-expansion candidates?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from phonepe_analytics.metrics import add_growth_metrics, add_ratio_metrics

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
CHART_DIR = ROOT / "reports" / "eda_charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

district = pd.read_csv(DATA_DIR / "district_quarter.csv")
state = pd.read_csv(DATA_DIR / "state_quarter.csv")
categories = pd.read_csv(DATA_DIR / "state_transaction_categories.csv")

district = add_ratio_metrics(district)
district = add_growth_metrics(district, ["state", "district"])

state = add_ratio_metrics(state)
state = add_growth_metrics(state, ["state"])

latest_period = district["period_id"].max()
latest_district = district[district["period_id"] == latest_period].copy()
latest_state = state[state["period_id"] == latest_period].copy()

latest_label = (
    f"{int(latest_district['year'].max())} "
    f"Q{int(latest_district['quarter'].max())}"
)

print("Latest period:", latest_label)
print("District rows:", f"{len(latest_district):,}")
print("States/UTs:", latest_district["state"].nunique())


## 1. Data quality and coverage

Start by checking the latest-quarter completeness and historical coverage. The expansion analysis should not rank districts that are missing the core demand or merchant variables.


In [ ]:
core_columns = [
    "transaction_count",
    "transaction_amount",
    "registered_users",
    "registered_merchants",
]

missing_values = latest_district[core_columns].isna().sum()

coverage = (
    district.groupby(["year", "quarter"])
    .agg(
        district_rows=("district", "size"),
        states=("state", "nunique"),
        districts=("district", "nunique"),
    )
    .reset_index()
)

display(missing_values.rename("missing_values").to_frame())
display(coverage.tail(8))

complete_rows = latest_district[core_columns].notna().all(axis=1).mean()

print(f"Complete latest-quarter district rows: {complete_rows:.1%}")
print(
    "Coverage period:",
    f"{coverage.iloc[0]['year']:.0f} Q{coverage.iloc[0]['quarter']:.0f}",
    "to",
    latest_label,
)


### Key insights

- The latest quarter is the main comparison period because it contains the current demand and merchant picture.
- The percentage of complete latest-quarter district rows tells us how much of the market can be compared without imputation.
- Historical merchant gaps should remain missing rather than being replaced with zero, because zero would incorrectly mean that no merchants existed.


## 2. Descriptive statistics

The next step is to understand the typical district, the spread between smaller and larger markets, and whether averages are distorted by a few very large districts.


In [ ]:
analysis_columns = [
    "transaction_count",
    "transaction_amount",
    "registered_users",
    "registered_merchants",
    "average_transaction_value",
    "merchants_per_100k_users",
    "users_per_merchant",
    "transaction_yoy",
]

descriptive_stats = (
    latest_district[analysis_columns]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .T
)

display(descriptive_stats)

median_users = latest_district["registered_users"].median()
median_merchants = latest_district["registered_merchants"].median()
median_penetration = latest_district["merchants_per_100k_users"].median()
median_growth = latest_district["transaction_yoy"].median()
positive_growth_share = latest_district["transaction_yoy"].gt(0).mean()

print(f"Median registered users: {median_users:,.0f}")
print(f"Median registered merchants: {median_merchants:,.0f}")
print(f"Median merchants per 100K users: {median_penetration:,.1f}")
print(f"Median transaction YoY growth: {median_growth:.1%}")
print(f"Districts with positive YoY transaction growth: {positive_growth_share:.1%}")


### Key insights

- Median values are more useful than means when district sizes are highly uneven.
- The median merchant-penetration rate provides a simple benchmark for identifying relatively under-penetrated districts.
- The share of districts with positive YoY transaction growth shows whether growth is broad-based or concentrated in only a few markets.


## 3. Univariate analysis

Visualize the four variables that matter most for the expansion decision: users, merchants, merchant penetration, and transaction growth.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(
    latest_district["registered_users"].dropna(),
    bins=30,
    kde=True,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Registered Users")

sns.histplot(
    latest_district["registered_merchants"].dropna(),
    bins=30,
    kde=True,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Registered Merchants")

sns.histplot(
    latest_district["merchants_per_100k_users"].dropna(),
    bins=30,
    kde=True,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Merchants per 100K Users")

sns.histplot(
    latest_district["transaction_yoy"].dropna(),
    bins=30,
    kde=True,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Transaction YoY Growth")

plt.tight_layout()
plt.savefig(CHART_DIR / "core_distributions.png", dpi=160)
plt.show()


### Key insights

- User and merchant counts are scale variables: large urban districts will naturally dominate absolute counts.
- Merchant penetration normalizes merchant registrations by the user base and allows fairer comparisons across districts.
- Growth should be interpreted together with market size because very small districts can show large percentage changes from a low base.


## 4. National payment trend

Before comparing regions, confirm whether the overall ecosystem is expanding or slowing down.


In [ ]:
national_trend = (
    state.groupby(["period_id", "year", "quarter"], as_index=False)
    .agg(
        transactions=("transaction_count", "sum"),
        transaction_value=("transaction_amount", "sum"),
        registered_users=("registered_users", "sum"),
        registered_merchants=("registered_merchants", "sum"),
    )
    .sort_values("period_id")
)

national_trend["period"] = (
    national_trend["year"].astype(str)
    + " Q"
    + national_trend["quarter"].astype(str)
)

display(national_trend.tail(8))

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=national_trend,
    x="period",
    y="transactions",
    marker="o",
)
plt.xticks(rotation=45, ha="right")
plt.title("National Transaction Trend")
plt.xlabel("")
plt.ylabel("Transactions")
plt.tight_layout()
plt.savefig(CHART_DIR / "national_transaction_trend.png", dpi=160)
plt.show()

latest_transactions = national_trend.iloc[-1]["transactions"]
previous_transactions = national_trend.iloc[-2]["transactions"]
latest_qoq = latest_transactions / previous_transactions - 1

print(f"Latest national transactions: {latest_transactions:,.0f}")
print(f"Latest national QoQ transaction growth: {latest_qoq:.1%}")


### Key insights

- National growth provides the context for regional growth. A district growing quickly during a strong national expansion is different from a district outperforming a weak national market.
- The latest QoQ national growth rate acts as a useful benchmark when reviewing district momentum.


## 5. State-level demand and concentration

Identify the largest state markets and measure how much of total activity is concentrated in the leaders.


In [ ]:
state_rank = latest_state.sort_values(
    "transaction_count",
    ascending=False,
).copy()

state_rank["transaction_share"] = (
    state_rank["transaction_count"]
    / state_rank["transaction_count"].sum()
)

top_states = state_rank[
    [
        "state",
        "transaction_count",
        "transaction_amount",
        "registered_users",
        "registered_merchants",
        "transaction_share",
    ]
].head(15)

display(top_states)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_states,
    x="transaction_count",
    y="state",
)
plt.title(f"Top States by Transaction Volume | {latest_label}")
plt.xlabel("Transactions")
plt.ylabel("")
plt.tight_layout()
plt.savefig(CHART_DIR / "top_states_transactions.png", dpi=160)
plt.show()

top_5_share = state_rank.head(5)["transaction_share"].sum()
top_10_share = state_rank.head(10)["transaction_share"].sum()

print(f"Top 5 states' share of transactions: {top_5_share:.1%}")
print(f"Top 10 states' share of transactions: {top_10_share:.1%}")
print("Largest state:", state_rank.iloc[0]["state"])


### Key insights

- The top-state concentration shows how much of PhonePe activity is concentrated in a small number of markets.
- High-volume states are strategically important, but volume leadership alone does not prove a merchant-acquisition gap.
- District-level analysis is necessary because the strongest expansion opportunities can sit inside both large and mid-sized states.


## 6. District-level demand, growth, and merchant penetration

Compare districts through three different lenses instead of relying on one ranking.


In [ ]:
columns = [
    "state",
    "district",
    "transaction_count",
    "registered_users",
    "registered_merchants",
    "transaction_yoy",
    "merchants_per_100k_users",
]

largest_districts = latest_district.nlargest(
    10,
    "transaction_count",
)[columns]

fastest_growing = (
    latest_district[latest_district["registered_users"] >= 100_000]
    .dropna(subset=["transaction_yoy"])
    .nlargest(10, "transaction_yoy")[columns]
)

lowest_penetration = (
    latest_district[
        (latest_district["registered_users"] >= 100_000)
        & (latest_district["registered_merchants"] > 0)
    ]
    .nsmallest(10, "merchants_per_100k_users")[columns]
)

print("Largest districts by transaction volume")
display(largest_districts)

print("Fastest-growing sizeable districts")
display(fastest_growing)

print("Lowest merchant penetration among sizeable districts")
display(lowest_penetration)

largest_names = set(largest_districts["district"])
growth_names = set(fastest_growing["district"])
penetration_names = set(lowest_penetration["district"])

repeated_names = (
    (largest_names & growth_names)
    | (largest_names & penetration_names)
    | (growth_names & penetration_names)
)

print("Districts appearing in at least two top-10 views:")
print(sorted(repeated_names))


### Key insights

- The largest districts, fastest-growing districts, and lowest-penetration districts are not the same group.
- Districts appearing in more than one list deserve closer attention because they combine more than one favorable signal.
- This cross-check prevents the project from treating a single extreme metric as enough evidence for expansion.


## 7. User growth versus merchant growth

A useful expansion signal is whether the registered-user base is growing faster than the registered-merchant base.


In [ ]:
growth_gap = latest_district.dropna(
    subset=["registered_users_qoq", "registered_merchants_qoq"]
).copy()

growth_gap["user_minus_merchant_growth"] = (
    growth_gap["registered_users_qoq"]
    - growth_gap["registered_merchants_qoq"]
)

growth_gap_share = (
    growth_gap["user_minus_merchant_growth"] > 0
).mean()

top_growth_gaps = (
    growth_gap[growth_gap["registered_users"] >= 100_000]
    .nlargest(15, "user_minus_merchant_growth")
    [
        [
            "state",
            "district",
            "registered_users",
            "registered_merchants",
            "registered_users_qoq",
            "registered_merchants_qoq",
            "user_minus_merchant_growth",
        ]
    ]
)

display(top_growth_gaps)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=growth_gap,
    x="registered_users_qoq",
    y="registered_merchants_qoq",
    alpha=0.5,
)
plt.axline((0, 0), slope=1, linestyle="--", linewidth=1)
plt.title("User Growth vs Merchant Growth")
plt.xlabel("Registered User QoQ Growth")
plt.ylabel("Registered Merchant QoQ Growth")
plt.tight_layout()
plt.savefig(CHART_DIR / "user_vs_merchant_growth.png", dpi=160)
plt.show()

print(
    "Districts where user growth exceeds merchant growth:",
    f"{growth_gap_share:.1%}",
)


### Key insights

- A positive growth gap means the registered-user base is expanding faster than merchant registrations.
- These markets are worth investigating because demand-side reach may be developing faster than merchant-side coverage.
- The signal is not proof of merchant shortage; active merchant counts and competitor acceptance would still need field validation.


## 8. Demand growth versus merchant penetration

This is the most direct exploratory view of the business problem: strong transaction growth combined with below-median merchant penetration.


In [ ]:
relationship_data = latest_district.dropna(
    subset=[
        "registered_users",
        "transaction_count",
        "transaction_yoy",
        "merchants_per_100k_users",
    ]
).copy()

growth_median = relationship_data["transaction_yoy"].median()
penetration_median = relationship_data[
    "merchants_per_100k_users"
].median()

relationship_data["high_growth"] = (
    relationship_data["transaction_yoy"] >= growth_median
)

relationship_data["low_penetration"] = (
    relationship_data["merchants_per_100k_users"]
    < penetration_median
)

exploration_candidates = relationship_data[
    relationship_data["high_growth"]
    & relationship_data["low_penetration"]
    & (relationship_data["registered_users"] >= 100_000)
].copy()

exploration_candidates = exploration_candidates.sort_values(
    ["transaction_yoy", "registered_users"],
    ascending=[False, False],
)

display(
    exploration_candidates[
        [
            "state",
            "district",
            "registered_users",
            "transaction_count",
            "transaction_yoy",
            "registered_merchants",
            "merchants_per_100k_users",
        ]
    ].head(20)
)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=relationship_data,
    x="merchants_per_100k_users",
    y="transaction_yoy",
    size="registered_users",
    sizes=(20, 200),
    alpha=0.5,
    legend=False,
)
plt.axvline(
    penetration_median,
    linestyle="--",
    linewidth=1,
)
plt.axhline(
    growth_median,
    linestyle="--",
    linewidth=1,
)
plt.title("Transaction Growth vs Merchant Penetration")
plt.xlabel("Merchants per 100K Users")
plt.ylabel("Transaction YoY Growth")
plt.tight_layout()
plt.savefig(CHART_DIR / "growth_vs_merchant_penetration.png", dpi=160)
plt.show()

candidate_share = (
    len(exploration_candidates)
    / len(relationship_data)
)

candidate_states = (
    exploration_candidates.groupby("state")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print(
    "High-growth, low-penetration sizeable districts:",
    len(exploration_candidates),
)
print(
    "Share of comparable districts:",
    f"{candidate_share:.1%}",
)
print("States with the most exploratory candidates:")
display(candidate_states.rename("candidate_districts").to_frame())


### Key insights

- The upper-left area of the chart represents the most interesting exploratory pattern: above-median growth with below-median merchant penetration.
- The candidate count tells us whether the opportunity pattern is rare or widespread.
- State concentration among these candidates can help prioritize where field teams should investigate first.
- These are screening candidates, not final recommendations. The formal opportunity score adds scale, intensity, and sensitivity checks.


## 9. Relationship strength

Use simple rank correlations to quantify the most important relationships without making causal claims.


In [ ]:
users_transactions_corr = latest_district[
    ["registered_users", "transaction_count"]
].corr(method="spearman").iloc[0, 1]

merchants_transactions_corr = latest_district[
    ["registered_merchants", "transaction_count"]
].corr(method="spearman").iloc[0, 1]

penetration_growth_corr = latest_district[
    ["merchants_per_100k_users", "transaction_yoy"]
].corr(method="spearman").iloc[0, 1]

correlations = pd.DataFrame(
    {
        "relationship": [
            "Registered users vs transactions",
            "Registered merchants vs transactions",
            "Merchant penetration vs transaction growth",
        ],
        "spearman_correlation": [
            users_transactions_corr,
            merchants_transactions_corr,
            penetration_growth_corr,
        ],
    }
)

display(correlations)


### Key insights

- A strong users-versus-transactions relationship confirms that market scale is an important driver of observed transaction volume.
- A strong merchants-versus-transactions relationship is expected in larger markets, but it does not prove that merchant registration causes transaction growth.
- The penetration-versus-growth relationship helps show whether lower-penetration markets systematically grow faster or whether opportunity is concentrated in specific districts.


## 10. Transaction category context

Check the latest state-level transaction mix so that total transaction growth is not incorrectly described as merchant-payment growth.


In [ ]:
latest_category_period = categories["period_id"].max()

latest_categories = categories[
    categories["period_id"] == latest_category_period
]

category_mix = (
    latest_categories.groupby("category", as_index=False)
    ["transaction_count"]
    .sum()
    .sort_values("transaction_count", ascending=False)
)

category_mix["transaction_share"] = (
    category_mix["transaction_count"]
    / category_mix["transaction_count"].sum()
)

display(category_mix)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=category_mix,
    x="transaction_share",
    y="category",
)
plt.title(f"Transaction Category Mix | {latest_label}")
plt.xlabel("Share of Transactions")
plt.ylabel("")
plt.tight_layout()
plt.savefig(CHART_DIR / "transaction_category_mix.png", dpi=160)
plt.show()

largest_category = category_mix.iloc[0]

print(
    "Largest transaction category:",
    largest_category["category"],
)
print(
    "Largest category share:",
    f"{largest_category['transaction_share']:.1%}",
)


### Key insights

- The category mix is important because PhonePe district transaction totals include more than merchant payments.
- Total transaction growth should therefore be described as digital-payment demand or ecosystem activity, not direct merchant sales.
- This limitation is one reason the final recommendation remains a merchant-expansion prioritization framework rather than a causal ROI model.


## 11. Final EDA findings

The EDA should lead directly into the opportunity-scoring notebook.

The main conclusions to carry forward are:

1. **Market scale matters.** Large user bases are strongly associated with transaction activity.
2. **Absolute merchant counts are not enough.** Merchant penetration needs to be normalized by the user base.
3. **Growth and penetration should be considered together.** High growth with relatively low merchant penetration is the core exploratory opportunity signal.
4. **User growth can expose additional gaps.** Districts where user registrations are growing faster than merchant registrations deserve investigation.
5. **No single metric should decide expansion.** Strong candidates should combine scale, demand, growth, and relatively low merchant penetration.
6. **Transaction totals are ecosystem-wide.** They must not be interpreted as merchant-only transaction activity.
7. **EDA identifies candidates, not causal opportunities.** The next step is the formal opportunity score, sensitivity analysis, and business segmentation.
